# Titanic Univariate Analysis

this notebook uses the datasets generated by `data/gen_uni.ipynb`. run that notebook first if the CSV files in `data/` are missing.

In [ ]:
import numpy as np
import pandas as pd

from missingfcup import MissingData
from missingfcup import Panel

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
df_mcar = pd.read_csv("data/mcar_uni.csv")
df_mar  = pd.read_csv("data/mar_uni.csv")
df_mnar = pd.read_csv("data/mnar_uni.csv")

In [13]:
md_mcar = MissingData(df_mcar)
md_mar = MissingData(df_mar)
md_mnar = MissingData(df_mnar)

## Exploratory Discovery

All three datasets were generated with the same missingness rate by design (20%, 142 missing values in age). These plots confirm the generation worked correctly. 

In [14]:
panel = Panel(
    plots=[
        md_mcar.barchart_overall_missingness(title="MCAR"),
        md_mar.barchart_overall_missingness(title="MAR"),
        md_mnar.barchart_overall_missingness(title="MNAR")
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

In [15]:
panel = Panel(
    plots=[
        md_mcar.barchart_missing_count(title="MCAR"),
        md_mar.barchart_missing_count(title="MAR"),
        md_mnar.barchart_missing_count(title="MNAR")
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

In [16]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missing_rate(title="MCAR"),
        md_mar.heatmap_missing_rate(title="MAR"),
        md_mnar.heatmap_missing_rate(title="MNAR")
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

## Mechanism Discovery

We now attempt to prove each mechanism using visualizations. The strategy used per case will be:

- MCAR: show that there are no observed variable that predict age missingness.
- MAR: show that pclass, explains age missingness (we know this beforehand but still). All missing values in age should locate at pclass=3.
- MNAR: First, we need to discard MAR by showing that there are no observed variable that can predict the missingness. Then, show that the age distribution itself explains its own missigness.

The following plot measures how strongly the observed values of each column can explain if age is missing.

In [30]:
panel = Panel(
    plots=[
        md_mcar.heatmap_value_missingness_correlation(title="MCAR"),
        md_mar.heatmap_value_missingness_correlation(title="MAR"),
        md_mnar.heatmap_value_missingness_correlation(title="MNAR")
        ],
    title="Missing Values by Mechanism",
    )

panel.show()

- MCAR: All correlations are near zero, so no observed variable could explain the missigness in others, so its random. First evidence of MCAR.

- MAR: pclass shows a strong positive relation with age missingness, confirming what we expected. First strong evidence in favour of MAR.

- MNAR: sibsp and parch show some positive values. This does not mean the mechanism is MAR. (We know that the MNAR generator removed young ages, and young passengers tend to travel with family, so sibsp and parch appear with missingness as a side effect). We need further plots to discard MAR and confirm MNAR.

Since pclass is the strongest candidate to explain MAR, we will use the heatmap_missingness ordered by pclass to test whether missing age values locate at specific pclass.

In [29]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missingness(
            title="MCAR", 
            order_by=[{"column": "pclass", "direction": "asc"}]
            ),
        md_mar.heatmap_missingness(
            title="MAR", 
            order_by=[{"column": "pclass", "direction": "asc"}]
            ),
        md_mnar.heatmap_missingness(
            title="MNAR",
            order_by=[{"column": "pclass", "direction": "asc"}]
            )
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

- MCAR: Missing age values are distributed throughout all rows regardless of pclass value. No pattern, still MCAR.

- MAR: Missing age values locate at the top of the heatmap, exactly where pclass=3 rows are sorted. Strong evidence for MAR.

- MNAR: Same situation as with MCAR. We will test with sibsp and parch, since they were potential candiates as well.

In [31]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missingness(
            title="MCAR", 
            order_by=[{"column": "sibsp", "direction": "asc"}]
            ),
        md_mar.heatmap_missingness(
            title="MAR", 
            order_by=[{"column": "sibsp", "direction": "asc"}]
            ),
        md_mnar.heatmap_missingness(
            title="MNAR",
            order_by=[{"column": "sibsp", "direction": "asc"}]
            )
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

- MCAR: No pattern.

- MAR: No pattern. Expected since for this dataset sibsp was not an important predictor.

- MNAR: Slightly more missing values at higher sibsp, but the pattern is not so clear and there are values across the full range. This removes the possibility of being MAR. 

In [32]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missingness(
            title="MCAR", 
            order_by=[{"column": "parch", "direction": "asc"}]
            ),
        md_mar.heatmap_missingness(
            title="MAR", 
            order_by=[{"column": "parch", "direction": "asc"}]
            ),
        md_mnar.heatmap_missingness(
            title="MNAR",
            order_by=[{"column": "parch", "direction": "asc"}]
            )
        ],
    title="Missing Value Counts by Mechanism",
    )

panel.show()

- MCAR: No pattern

- MAR: No pattern

- MNAR: Similar situation, tendency to have the missing values at higher parch values, but not so clear to conclude MAR. We can say that sibsp and parch are potential MNAR, since there is a relation between them and age. Its not random (not MCAR), but also those variables do not explain missigness (so not MAR), we can say they are a result of the removal but not the reason behind it.

In [ ]:
all_pclass = pd.concat([df_mcar["pclass"], df_mar["pclass"], df_mnar["pclass"]]).dropna()
all_age    = pd.concat([df_mcar["age"],    df_mar["age"],    df_mnar["age"]]).dropna()

age_min    = all_age.min()
age_offset = age_min - 0.12 * (all_age.max() - age_min)
yaxis_range = [age_offset - 3, all_age.max() * 1.05]
xaxis_range = [all_pclass.min() - 0.5, all_pclass.max() + 0.5]

panel = Panel(
    plots=[
        md_mcar.scatterplot_missingness(x="pclass", y="age", title="MCAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        md_mar.scatterplot_missingness(x="pclass", y="age", title="MAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        md_mnar.scatterplot_missingness(x="pclass", y="age", title="MNAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        ],
    title="Age vs Passenger Class by Missingness Mechanism",
    )

panel.show()

- MCAR: Red triangles (missing age) spread across all values of pclass. No pattern. MCAR confirmed.

- MAR All red triangles are locked at pclass=3. MAR confirmed.

- MNAR Red triangles spread across all pclass values, no pattern there. We can see that the present values start to be shown on age=15 rather than age=0 as in MCAR and MAR. This is because the values of younger people were removed. This is proof of MNAR. The missing values don't have a predictor and there is a clear pattern on its own column. MNAR confirmed.

## Comparative Discovery

The heatmaps confirm pclass as the MAR predictor and rule it out for MNAR. We now use boxplot and density to double check.

In [21]:
panel = Panel(
    plots=[
        md_mcar.boxplot_missingness(x="pclass", color_by="age", title="MCAR"),
        md_mar.boxplot_missingness(x="pclass", color_by="age", title="MAR"),
        md_mnar.boxplot_missingness(x="pclass", color_by="age", title="MNAR")
        ],
    title="pclass Distribution split by Age Missingness",
    )

panel.show()

Boxplot might not be the best plot for pclass, but we managed to get some interesting insights:

- MCAR: We can see age present values spread across all values of p class, and for the missing cases. No explicit pattern.

- MAR: The "age: missing" box is a line at pclass=3, this is because all 142 missing age values are pclass=3. And the age present values are present accross all classes. MAR confirmed.

- MNAR: Both boxes are similar to MCAR, pclass does not show a pattern for missing values.

In [33]:
panel = Panel(
    plots=[
        md_mcar.density_missingness(x="pclass", color_by="age", title="MCAR"),
        md_mar.density_missingness(x="pclass", color_by="age", title="MAR"),
        md_mnar.density_missingness(x="pclass", color_by="age", title="MNAR")
        ],
    title="pclass Density split by Age Missingness",
    )

panel.show()

- MCAR: The !NA and NA density curves almost overlap across all pclass values, so no pattern. Confirmation of MCAR.

- MAR: The !NA curve has density across all three pclass values. The NA curve is not visible, because all 142 missing values are exactly pclass=3, so no variance, no curve. The fact that there is no NA curve is the proof, that missing age values exist only at one pclass level. Confirmation of MAR

- MNAR: Both curves overlap, similar to MCAR. No pattern.

## Conclusions

- MCAR: No observed variable explain age missingness. Missing values spread uniformly across all variables and all pclass values. The correlation heatmap also didn't show any potential predictors, heatmap ordered by pclass/sibsp/parch didn't show patterns, boxplot show identical distributions

- MAR: pclass predicts age missingness since all 142 missing values belong to pclass=3. No other predictor is found. The correlation heatmap showed pclass as only serious predictor, heatmap ordered by pclass showed a pattern at the top, and boxplot showed a flat line at pclass=3, density didn't have a curve, and scatter showed all red dots at pclass=3.

- MNAR: No observed variable predicts the missingness, so discarded MAR. The observed age distribution showed that lower values are missing, showing that there is a pattern. Correlations with sibsp and parch are results of young ages being removed.

In [18]:
# almost no informative when only one column has missing values
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# md_mcar.barchart_intersection(title="MCAR").show()
# md_mar.barchart_intersection(title="MAR").show()
# md_mnar.barchart_intersection(title="MNAR").show()

In [19]:
# almost no informative when only one column has missing values
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# panel = Panel(
#     plots=[
#         md_mcar.barchart_venn(title="MCAR"),
#         md_mar.barchart_venn(title="MAR"),
#         md_mnar.barchart_venn(title="MNAR")
#         ],
#     title="Missing Value Counts by Mechanism",
#     )

# panel.show()

In [22]:
# dendogram needs more columns to compute missingness patterns
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# panel = Panel(
#     plots=[
#         md_mcar.dendrogram_missingness(title="MCAR"),
#         md_mar.dendrogram_missingness(title="MAR"),
#         md_mnar.dendrogram_missingness(title="MNAR")
#         ],
#     title="Missing Value Counts by Mechanism",
#     )

# panel.show()

In [24]:
# this correlation heatmap_correlation_missing_missing needs more columns to compute missingness patterns
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# panel = Panel(
#     plots=[
#         md_mcar.heatmap_correlation_missing_missing(title="MCAR"),
#         md_mar.heatmap_correlation_missing_missing(title="MAR"),
#         md_mnar.heatmap_correlation_missing_missing(title="MNAR")
#         ],
#     title="Missing Value Counts by Mechanism",
#     )

# panel.show()

In [25]:
# almost no informative when only one column has missing values
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# panel = Panel(
#     plots=[
#         md_mcar.heatmap_correlation_present_missing(title="MCAR"),
#         md_mar.heatmap_correlation_present_missing(title="MAR"),
#         md_mnar.heatmap_correlation_present_missing(title="MNAR")
#         ],
#     title="Missing Value Counts by Mechanism",
#     )

# panel.show()

In [26]:
# this correlation heatmap_correlation_present_present needs more columns to compute missingness patterns
# we will leave it out of this analysis for now, but will be useful on multivariate analysis when we have more columns to analyze

# panel = Panel(
#     plots=[
#         md_mcar.heatmap_correlation_present_present(title="MCAR"),
#         md_mar.heatmap_correlation_present_present(title="MAR"),
#         md_mnar.heatmap_correlation_present_present(title="MNAR")
#         ],
#     title="Missing Value Counts by Mechanism",
#     )

# panel.show()